# Week 2 · What ML is, and linear regression

Machine learning starts here. This notebook follows the first two topics of the ML course syllabus: **introduction to ML** and **linear regression**, including ridge and lasso. Every formula is derived step by step, and there is one worked example with every number computed. You've already built most of the pieces: the least-squares gradient and the normal equation ([Week 1 · math §7 ↗](../week1/03_math_essentials.ipynb)), projections ([notebook 01 ↗](01_linear_algebra_review.ipynb)), and MAP = ridge ([notebook 02 ↗](02_map_and_entropy_in_depth.ipynb)). **Reading:** ISL Ch. 2 (what statistical learning is), Ch. 3.1–3.2 (linear regression), Ch. 6.2 (ridge and lasso).

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **feature**: One input variable, e.g. a house's size. A sample's features form the vector $x \in \mathbb{R}^d$.
- **label / target**: What we want to predict, $y$.
- **model $f_w$**: A function with adjustable parameters $w$ that maps features to a prediction $\hat y = f_w(x)$.
- **loss**: A number $L(w)$ that measures how wrong the predictions are on the data.
- **training**: Choosing $w$ to make the loss small, i.e. $\hat w = \arg\min_w L(w)$.
- **residual**: $e_i = y_i - \hat y_i$, what the model missed on sample $i$.
- **regularization**: Adding a penalty on the size of $w$ to the loss, so the model can't fit noise as easily.

## 1. What machine learning is

**Machine learning means choosing a function from data instead of writing it by hand.** You pick a family of functions (the **model**), a way to measure mistakes (the **loss**), and a procedure that finds the best member of the family (the **optimizer**). Every algorithm in this course is one choice of these three:

| Algorithm | Model | Loss | Optimizer |
|---|---|---|---|
| linear regression | $\hat y = w^\top x + b$ | squared error | normal equation, or gradient descent |
| logistic regression (`05`) | $\hat p = \sigma(w^\top x + b)$ | cross-entropy | gradient descent |
| neural network (Week 5) | layers of $\sigma(Wx + b)$ | cross-entropy or MSE | gradient descent (backprop) |
| k-means (Week 4) | k cluster centers | within-cluster squared distance | alternate assign / update |

**Two big families.** In **supervised** learning every training sample comes with the answer $y$: in **regression** $y$ is a number (a price, a temperature), and in **classification** $y$ is a category (spam or not). In **unsupervised** learning there's no $y$, and the goal is to find structure, like clusters or the main directions of variation (Week 4).

**Parametric vs non-parametric.** Linear regression is **parametric**: everything it learned fits in a fixed set of numbers $w$, no matter how much data there is. k-nearest neighbors (`05`) is **non-parametric**: it keeps the whole training set and looks at it at prediction time.

**The real goal is the test error, not the training error.** We train on data we have, but we care about new data the model hasn't seen. That's why we keep a **test set** aside ([Week 1 · math §8 ↗](../week1/03_math_essentials.ipynb): train for fitting, validation for choosing settings, test once at the end). A model that is great on training data but bad on the test set has **overfit**: it learned the noise.

**Quick check:** Predicting tomorrow's electricity peak (in kW) from today's load: regression or classification? And predicting whether tomorrow will have a peak alert (yes/no)?

<details><summary>Answer</summary>

The first is regression, because the target is a number. The second is classification, because the target is a category. Your KAMP project did both.

</details>

## 2. The linear regression model

**Linear regression predicts a number as a weighted sum of the features plus a constant.**

**The setting.** $n$ samples, each with $d$ features $x_i \in \mathbb{R}^d$ and a numeric target $y_i$. The model has one weight per feature, $w \in \mathbb{R}^d$, plus an intercept $b$:

$$
\hat y_i = w^\top x_i + b = w_1 x_{i1} + w_2 x_{i2} + \dots + w_d x_{id} + b
$$

Here $w_j$ is how much the prediction changes when feature $j$ increases by 1 (all else fixed), and $b$ is the prediction when every feature is 0.

**The intercept trick.** Add a column of 1s to the data. Then $b$ becomes just another weight, and the whole model is one matrix product $\hat y = Xw$, with $X \in \mathbb{R}^{n \times (d+1)}$:

$$
X = \begin{pmatrix} 1 & x_{11} & \cdots & x_{1d} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{nd} \end{pmatrix}, \qquad w = (b, w_1, \dots, w_d)
$$

**The loss** is the mean squared error. Squaring makes every miss count as positive, punishes big misses most, and is what MLE gives under Gaussian noise ([Week 1 · math §4 ↗](../week1/03_math_essentials.ipynb): minimizing the MSE is the same as maximizing the likelihood when $y = \hat y + \varepsilon$ with Gaussian $\varepsilon$):

$$
L(w) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat y_i)^2 = \frac{1}{n} \lVert y - Xw \rVert^2
$$

## 3. Fitting one feature by hand

**With one feature, least squares has a closed form you can compute on paper.**

**Where it comes from.** Minimize $S(w, b) = \sum_i (y_i - b - w x_i)^2$ by setting both partial derivatives to 0.

**Step 1, the intercept.** Take $\frac{\partial S}{\partial b} = -2 \sum_i (y_i - b - w x_i) = 0$ and divide by $-2n$:

$$
\bar y - b - w \bar x = 0 \;\Longrightarrow\; b = \bar y - w \bar x
$$

So **the fitted line always passes through the point of means** $(\bar x, \bar y)$.

**Step 2, the slope.** Put $b = \bar y - w\bar x$ into $\frac{\partial S}{\partial w} = -2 \sum_i x_i (y_i - b - w x_i) = 0$. The term inside becomes $(y_i - \bar y) - w(x_i - \bar x)$, and solving for $w$ gives:

$$
\hat w = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2} \qquad \hat b = \bar y - \hat w\, \bar x
$$

In words: the slope is how much $x$ and $y$ move together, divided by how much $x$ moves on its own (the sample covariance over the sample variance).

**Worked example** (the same data as [notebook 01 §4 ↗](01_linear_algebra_review.ipynb)). $x = (1, 2, 3, 4)$, $y = (2, 3, 5, 4)$, so $\bar x = 2.5$ and $\bar y = 3.5$:

| $i$ | $x_i$ | $y_i$ | $x_i - \bar x$ | $y_i - \bar y$ | product | $(x_i - \bar x)^2$ |
|---|---|---|---|---|---|---|
| 1 | 1 | 2 | −1.5 | −1.5 | 2.25 | 2.25 |
| 2 | 2 | 3 | −0.5 | −0.5 | 0.25 | 0.25 |
| 3 | 3 | 5 | 0.5 | 1.5 | 0.75 | 0.25 |
| 4 | 4 | 4 | 1.5 | 0.5 | 0.75 | 2.25 |
| **sum** |  |  |  |  | **4** | **5** |

So $\hat w = \frac{4}{5} = 0.8$ and $\hat b = 3.5 - 0.8 \times 2.5 = 1.5$. The line is $\hat y = 1.5 + 0.8x$, exactly what the projection gave in [notebook 01 ↗](01_linear_algebra_review.ipynb).

| $x_i$ | $y_i$ | $\hat y_i = 1.5 + 0.8x_i$ | residual $e_i$ | $e_i^2$ |
|---|---|---|---|---|
| 1 | 2 | 2.3 | −0.3 | 0.09 |
| 2 | 3 | 3.1 | −0.1 | 0.01 |
| 3 | 5 | 3.9 | 1.1 | 1.21 |
| 4 | 4 | 4.7 | −0.7 | 0.49 |
| **sum** |  |  | **0** | **1.80** |

The residuals sum to exactly 0. That's not luck: it's the intercept equation from step 1, and the "error ⟂ the column of 1s" fact from [notebook 01 ↗](01_linear_algebra_review.ipynb).

In [3]:
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.0]); y = np.array([2.0, 3.0, 5.0, 4.0])
w = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b = y.mean() - w * x.mean()
print(w, b)                                   # → 0.8 1.5
X = np.column_stack([np.ones(4), x])          # the intercept trick
print(np.linalg.solve(X.T @ X, X.T @ y))      # → [1.5 0.8]   same answer from the normal equation
#np.linalg.solve(A, b) returns the w satisfying Aw = b, which is exactly the normal equation.

0.8 1.5
[1.5 0.8]


With the worked example's data and $w = (b, w_1) = (1.5, 0.8)$, here is what `X @ w` does, row by row:

```text
        X                 w                  X @ w
  [ 1   1 ]                                1·1.5 + 1·0.8 = 2.3
  [ 1   2 ]    @    [ 1.5 ]      =         1·1.5 + 2·0.8 = 3.1
  [ 1   3 ]         [ 0.8 ]                1·1.5 + 3·0.8 = 3.9
  [ 1   4 ]                                1·1.5 + 4·0.8 = 4.7
   ↑
   the 1 in every row multiplies b, so every prediction gets "+ b"
```

- Each row of `X` is one sample. `@` dots each row with `w`, giving one prediction per sample: shape $(4 \times 2)(2) \to (4)$.
- Row 3 is `[1, 3]`: $1 \cdot 1.5 + 3 \cdot 0.8 = 1.5 + 2.4 = 3.9$.
- The first column holds only 1s, so $b$ is added to every prediction, exactly like $\hat y = b + w_1 x$.

**Drill (on paper).** Fit a line by hand to $x = (0, 1, 2)$, $y = (1, 2, 6)$.

<details><summary>Worked answer</summary>

$\bar x = 1$, $\bar y = 3$. Deviations: $x - \bar x = (-1, 0, 1)$ and $y - \bar y = (-2, -1, 3)$. Products: $2, 0, 3$, sum $5$. Squares of the $x$ deviations: $1, 0, 1$, sum $2$. So $\hat w = 2.5$ and $\hat b = 3 - 2.5 = 0.5$: $\hat y = 0.5 + 2.5x$.

</details>

## 4. Gradient descent for linear regression

**Gradient descent finds the same line by taking repeated small downhill steps, and it still works when there are too many features to solve the normal equation.**

**How many features is "too many"?** There's no exact cutoff. "Too many" means the normal equation becomes too expensive in memory or time, and you can check that with arithmetic.

What the normal equation costs (with $n$ samples and $p$ features):

| Step | Time | Memory |
|---|---|---|
| Form $X^\top X$ | $O(np^2)$ | $p \times p$ matrix |
| Solve $(X^\top X)w = X^\top y$ | $O(p^3)$ | same matrix |

The $p^3$ and $p^2$ terms are the problem. Time grows with the cube of the feature count, and memory with the square.

Plugging in numbers (float64 = 8 bytes per entry):

| $p$ | Size of $X^\top X$ | Solve cost ($\sim p^3$ flops) | Verdict |
|---|---|---|---|
| 1,000 | 8 MB | $10^9$, under a second | trivial |
| 10,000 | 800 MB | $10^{12}$, seconds to a minute | fine on a laptop |
| 100,000 | 80 GB | $10^{15}$, hours or more | infeasible on most machines |
| 1,000,000 | 8 TB | $10^{18}$ | impossible |

So the rough rule is: **up to about $10^4$ features is fine, and around $10^5$ and beyond you switch to gradient descent.** In practice you hit the wall when $X^\top X$ doesn't fit in RAM.

**The gradients** of the MSE ([Week 1 · math §7 ↗](../week1/03_math_essentials.ipynb) derived $2X^\top(Xw - y)$ for the sum of squares; divide by $n$ because this loss is a mean):

$$
\frac{\partial L}{\partial w} = -\frac{2}{n} \sum_i x_i (y_i - \hat y_i) \qquad \frac{\partial L}{\partial b} = -\frac{2}{n} \sum_i (y_i - \hat y_i) \qquad\text{matrix form: } \nabla L = \frac{2}{n} X^\top (Xw - y)
$$

<details>
<summary><b>Derivation of the gradient (click to expand)</b></summary>

**The setup**

$$L(w, b) = \frac{1}{n}\sum_{i=1}^n (y_i - \hat y_i)^2, \qquad \hat y_i = w^\top x_i + b$$

**Derivative with respect to $w$**

Differentiate one term of the sum. By the chain rule (outer function $u^2$, inner $u = y_i - \hat y_i$):

$$\frac{\partial}{\partial w}(y_i - \hat y_i)^2 = 2(y_i - \hat y_i)\cdot\frac{\partial}{\partial w}(y_i - \hat y_i)$$

$y_i$ doesn't depend on $w$, and $\frac{\partial}{\partial w}(w^\top x_i + b) = x_i$, so

$$\frac{\partial}{\partial w}(y_i - \hat y_i) = -x_i$$

Putting it together and averaging over the sum:

$$\frac{\partial L}{\partial w} = \frac{1}{n}\sum_i 2(y_i - \hat y_i)(-x_i) = -\frac{2}{n}\sum_i x_i\,(y_i - \hat y_i)$$

**Derivative with respect to $b$**

The same, except $\frac{\partial}{\partial b}(w^\top x_i + b) = 1$, so the $x_i$ becomes $1$:

$$\frac{\partial L}{\partial b} = -\frac{2}{n}\sum_i (y_i - \hat y_i)$$

You can think of $b$ as a weight on a feature that's always 1.

**Matrix form**

Stack the samples as rows of $X$. Then $\hat y = Xw$ (with $b$ absorbed by adding a column of ones to $X$), and

$$\sum_i x_i\,(y_i - \hat y_i) = X^\top(y - Xw)$$

since $X^\top v$ is exactly "sum of each row $x_i$ times $v_i$." So

$$\nabla L = -\frac{2}{n}X^\top(y - Xw) = \frac{2}{n}X^\top(Xw - y)$$

Flipping the sign inside just moves the minus sign. You can also get this directly from $L = \frac{1}{n}\|Xw - y\|^2$ using $\nabla_w \|Xw - y\|^2 = 2X^\top(Xw - y)$, which is the normal-equation derivation divided by $n$.

</details>

**The first three steps** on the worked example, starting at $w = b = 0$ with learning rate $\eta = 0.05$:

| Step | $\partial L / \partial w$ | $\partial L / \partial b$ | new $w$ | new $b$ |
|---|---|---|---|---|
| 1 | −19.5 | −7.0 | 0.975 | 0.350 |
| 2 | −3.125 | −1.425 | 1.131 | 0.421 |
| 3 | −0.425 | −0.501 | 1.153 | 0.446 |

Step 1, by hand: every prediction is 0, so the errors are $\hat y - y = (-2, -3, -5, -4)$. Then $\frac{\partial L}{\partial w} = \frac{2}{4}(1 \cdot (-2) + 2 \cdot (-3) + 3 \cdot (-5) + 4 \cdot (-4)) = 0.5 \times (-39) = -19.5$, and $w = 0 - 0.05 \times (-19.5) = 0.975$.

In [4]:
def gd(lr, steps):
    w, b = 0.0, 0.0
    for _ in range(steps):
        err = (b + w * x) - y                 # predictions minus targets
        w -= lr * 2 * np.mean(err * x)
        b -= lr * 2 * np.mean(err)
    return round(float(b), 4), round(float(w), 4)

print(gd(0.05, 200))     # → (1.4461, 0.8183)   close, not there yet
print(gd(0.05, 2000))    # → (1.5, 0.8)         converged to the exact answer
print(gd(0.5, 50))       # learning rate too big: the numbers explode to about -1e43

(1.4461, 0.8183)
(1.5, 0.8)
(-8.254550461768534e+42, -2.4269384972670023e+43)


**What one pass through the loop computes, array by array**

The code below runs `err = (b + w * x) - y`, then two updates. Here is the very first pass of the loop above, $w = b = 0$, written out:

```text
x               =  [  1    2    3    4 ]
y               =  [  2    3    5    4 ]
b + w * x       =  [  0    0    0    0 ]      every prediction is 0 at the start
err = pred - y  =  [ -2   -3   -5   -4 ]
err * x         =  [ -2   -6  -15  -16 ]      element by element

w step:  2 * mean(err * x) = 2 * (-39 / 4) = -19.5    →  w = 0 - 0.05 * (-19.5) = 0.975
b step:  2 * mean(err)     = 2 * (-14 / 4) =  -7.0    →  b = 0 - 0.05 * (-7.0)  = 0.35
```

Both gradients are negative, so both updates *increase* $w$ and $b$: the line rises toward the data. The next pass starts from $w = 0.975$, $b = 0.35$.

**Why the learning rate matters:** too small and you need thousands of steps; too big and each step overshoots the bottom, so the loss grows instead of shrinking. **Feature scaling** helps too: if one feature is in the thousands and another is under 1, a single learning rate can't suit both. Standardize first ([Week 1 · NumPy §9 ↗](../week1/02_numpy_lesson.ipynb): `(X - X.mean(axis=0)) / X.std(axis=0)` gives every column mean 0 and standard deviation 1).

## 5. How good is the fit?

**Regression metrics summarize the residuals in one number.**

| Metric | Formula | Worked example | Read it as |
|---|---|---|---|
| MSE | $\frac{1}{n}\sum_i e_i^2$ | $\frac{1.8}{4} = 0.45$ | average squared miss (the training loss) |
| RMSE | $\sqrt{\text{MSE}}$ | $\sqrt{0.45} = 0.671$ | a typical miss, in the units of $y$ |
| MAE | $\frac{1}{n}\sum_i \lvert e_i \rvert$ | $\frac{0.3 + 0.1 + 1.1 + 0.7}{4} = 0.55$ | average absolute miss; less sensitive to one big miss |
| $R^2$ | $1 - \frac{\text{SSE}}{\text{SST}}$ | $1 - \frac{1.8}{5} = 0.64$ | fraction of the variance in $y$ the model explains |

Here $\text{SSE} = \sum_i e_i^2$ is what the model gets wrong, and $\text{SST} = \sum_i (y_i - \bar y)^2$ is how much $y$ varies in total (in the worked example, $2.25 + 0.25 + 2.25 + 0.25 = 5$). **$R^2 = 1$** means perfect predictions; **$R^2 = 0$** means no better than always predicting $\bar y$; on the test set, $R^2$ can even be negative.

In [5]:
yhat = 1.5 + 0.8 * x; e = y - yhat
print(round(np.mean(e**2), 2), round(np.sqrt(np.mean(e**2)), 3), round(np.mean(np.abs(e)), 2), round(1 - (e**2).sum() / ((y - y.mean())**2).sum(), 2))
# → 0.45 0.671 0.55 0.64

0.45 0.671 0.55 0.64


## 6. Ridge regression (L2)

**Ridge regression adds a penalty on the squared size of the weights, which shrinks them toward 0 and reduces overfitting.**

$$
L_{\text{ridge}}(w) = \sum_i (y_i - \hat y_i)^2 + \lambda \sum_{j=1}^{d} w_j^2 \qquad \hat w_{\text{ridge}} = (X^\top X + \lambda I)^{-1} X^\top y
$$

Here $\lambda \ge 0$ sets the strength of the penalty. **Where it comes from:** MAP with a Gaussian prior $w_j \sim \mathcal{N}(0, \tau^2)$, with $\lambda = \sigma^2 / \tau^2$, derived step by step in [notebook 02 §4 ↗](02_map_and_entropy_in_depth.ipynb). **Why it helps:** big weights let a model chase noise. Making them cost something trades a little bias for a big drop in variance (the bias–variance trade-off, [Week 1 · math §8 ↗](../week1/03_math_essentials.ipynb)).

**Two practical rules.** (1) **Standardize the features first**, otherwise the penalty hits features with small units harder. (2) **Don't penalize the intercept** $b$: shrinking it toward 0 would just shift every prediction. scikit-learn handles both details for the intercept. In scikit-learn, $\lambda$ is called `alpha`.

**Worked example** on scikit-learn's built-in diabetes data (442 patients, 10 standardized features, target = disease progression after one year). As `alpha` grows, the length of the weight vector $\lVert \hat w \rVert$ shrinks:

In [6]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, Lasso
X, Y = load_diabetes(return_X_y=True)
print(X.shape)                                                        # → (442, 10)
print(round(np.linalg.norm(LinearRegression().fit(X, Y).coef_), 1))   # → 1377.8   no penalty
for alpha in [0.01, 0.1, 1, 10]:
    print(alpha, round(np.linalg.norm(Ridge(alpha=alpha).fit(X, Y).coef_), 1))
# → 0.01 987.6
#   0.1 799.5
#   1 511.6
#   10 145.5

(442, 10)
1377.8
0.01 987.6
0.1 799.5
1 511.6
10 145.5


## 7. Lasso (L1)

**Lasso penalizes the absolute size of the weights, which pushes some of them to exactly 0, so it also selects features.**

$$
L_{\text{lasso}}(w) = \sum_i (y_i - \hat y_i)^2 + \lambda \sum_{j=1}^{d} \lvert w_j \rvert
$$

**Why exact zeros.** The penalty $\lvert w_j \rvert$ has a sharp corner at 0, so near 0 it keeps pulling with the same force $\lambda$ however small $w_j$ gets. A weight whose feature doesn't reduce the error by at least that much is pushed all the way to 0. Ridge's $w_j^2$ is flat at 0: its pull fades as $w_j \to 0$, so weights get small but never exactly zero. Lasso has no closed form; it's solved iteratively. It's MAP with a **Laplace** prior ([notebook 02 §4 ↗](02_map_and_entropy_in_depth.ipynb)).

In [ ]:
for alpha in [0.01, 0.1, 1, 5]:
    coef = Lasso(alpha=alpha).fit(X, Y).coef_
    print(alpha, int((coef == 0).sum()), "of 10 weights are exactly 0")
# → 0.01 0 of 10 weights are exactly 0
#   0.1 3 of 10 weights are exactly 0
#   1 7 of 10 weights are exactly 0
#   5 10 of 10 weights are exactly 0

|  | Ridge (L2) | Lasso (L1) |
|---|---|---|
| penalty | $\lambda \sum_j w_j^2$ | $\lambda \sum_j \lvert w_j \rvert$ |
| effect | shrinks all weights smoothly | shrinks, and sets some weights to exactly 0 |
| solution | closed form $(X^\top X + \lambda I)^{-1} X^\top y$ | iterative only |
| prior (MAP view) | Gaussian | Laplace |
| use when | many features, each contributing a little; correlated features | you suspect only a few features matter |

**Quick check:** What happens to both ridge and lasso as $\lambda \to 0$?

<details><summary>Answer</summary>

Both become ordinary least squares.

</details>

---
Next: `05_logistic_regression_knn.ipynb` (classification). Exercises 1–5 in `07_exercises.ipynb` cover this notebook.